# MLflow Experiment Tracking

## Objective

Track the Flight Price Prediction model using MLflow.

This notebook will:

1. Load processed data
2. Train a Random Forest model
3. Evaluate model performance
4. Log parameters and metrics to MLflow
5. Save model artifacts

#### Code Cell

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("MLflow Version:", mlflow.__version__)

MLflow Version: 2.17.2


In [2]:
# Create folders if they don't exist

os.makedirs("../models", exist_ok=True)
os.makedirs("../mlruns", exist_ok=True)

In [3]:
# Load Dataset

flight_user = pd.read_csv(
    "../data/processed/flight_user.csv"
)

print("Shape:", flight_user.shape)

flight_user.head()

Shape: (271888, 19)


,travelCode,userCode,from,to,flightType,price,time,distance,agency,date,year,month,day,day_of_week,code,company,name,gender,age
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019-09-26,2019,9,26,Thursday,0,4You,Roy Braun,male,21
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019-09-30,2019,9,30,Monday,0,4You,Roy Braun,male,21
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,2019-10-03,2019,10,3,Thursday,0,4You,Roy Braun,male,21
3,1,0,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,2019-10-04,2019,10,4,Friday,0,4You,Roy Braun,male,21
4,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,2019-10-10,2019,10,10,Thursday,0,4You,Roy Braun,male,21


In [4]:
# Feature Selection

feature_columns = [
    "distance",
    "time",
    "flightType",
    "agency",
    "from",
    "to",
    "month"
]

X = flight_user[feature_columns]

y = flight_user["price"]

In [5]:
# Encode Categorical Variables

X = pd.get_dummies(
    X,
    columns=[
        "flightType",
        "agency",
        "from",
        "to"
    ],
    drop_first=True
)

print(X.shape)

(271888, 23)


In [6]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

Train Shape: (217510, 23)
Test Shape : (54378, 23)


In [7]:
# Create MLflow Experiment

mlflow.set_experiment(
    "Flight Price Prediction"
)

<Experiment: artifact_location='file:///c:/Projects/Github/Voyage_Analytics_MLOps_Project/notebooks/mlruns/271068965275279615', creation_time=1780762289697, experiment_id='271068965275279615', last_update_time=1780762289697, lifecycle_stage='active', name='Flight Price Prediction', tags={}>

In [8]:
with mlflow.start_run():

    # Train Model
    rf = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    rf.fit(
        X_train,
        y_train
    )

    # Predictions
    predictions = rf.predict(
        X_test
    )

    # Metrics
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    # Log Parameters
    mlflow.log_param(
        "model_type",
        "RandomForestRegressor"
    )

    mlflow.log_param(
        "n_estimators",
        100
    )

    mlflow.log_param(
        "random_state",
        42
    )

    # Log Metrics
    mlflow.log_metric(
        "MAE",
        mae
    )

    mlflow.log_metric(
        "RMSE",
        rmse
    )

    mlflow.log_metric(
        "R2",
        r2
    )

    # Save Model
    joblib.dump(
        rf,
        "../models/flight_price_model.pkl"
    )

    # Log Model
    mlflow.sklearn.log_model(
        rf,
        artifact_path="flight_price_model"
    )

    # Log Artifact
    mlflow.log_artifact(
        "../models/flight_price_model.pkl"
    )

    print("=" * 50)
    print("RUN COMPLETED")
    print("=" * 50)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")

2026/06/06 21:49:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RUN COMPLETED
MAE  : 0.0000
RMSE : 0.0000
R2   : 1.0000


In [12]:
# View Runs

runs = mlflow.search_runs()

runs[
    [
        "run_id",
        "metrics.MAE",
        "metrics.RMSE",
        "metrics.R2"
    ]
]

,run_id,metrics.MAE,metrics.RMSE,metrics.R2
0,60e284184ef247e2a86a2b2db4cddba8,2.874319e-12,4.131942e-12,1.0
1,73dd46d0be294acda437105693baf5f4,NaN,NaN,NaN


In [10]:
# Feature Importance

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(15)

,Feature,Importance
3,flightType_firstClass,0.296597
1,time,0.251551
0,distance,0.230820
4,flightType_premium,0.076931
9,from_Florianopolis (SC),0.051412
17,to_Florianopolis (SC),0.018754
14,from_Sao Paulo (SP),0.014580
7,from_Brasilia (DF),0.013291
22,to_Sao Paulo (SP),0.006796
18,to_Natal (RN),0.006589


In [11]:
# Save Feature Importance

feature_importance.to_csv(
    "../models/feature_importance.csv",
    index=False
)

print("Feature Importance Saved")

Feature Importance Saved


In [13]:
import mlflow

runs = mlflow.search_runs()

print("Number of runs:", len(runs))
runs

Number of runs: 2


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.R2,metrics.RMSE,metrics.MAE,params.model_type,params.random_state,params.n_estimators,tags.mlflow.source.name,tags.mlflow.log-model.history,tags.mlflow.source.git.commit,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user
0,60e284184ef247e2a86a2b2db4cddba8,271068965275279615,FINISHED,file:///c:/Projects/Github/Voyage_Analytics_ML...,2026-06-06 16:17:43.192000+00:00,2026-06-06 16:19:58.602000+00:00,1.0,4.131942e-12,2.874319e-12,RandomForestRegressor,42,100,c:\Projects\Github\Voyage_Analytics_MLOps_Proj...,"[{""run_id"": ""60e284184ef247e2a86a2b2db4cddba8""...",8a9354e0c89303995172ede154acac958c08451e,LOCAL,nervous-snake-400,Ganesh
1,73dd46d0be294acda437105693baf5f4,271068965275279615,FAILED,file:///c:/Projects/Github/Voyage_Analytics_ML...,2026-06-06 16:11:37.197000+00:00,2026-06-06 16:11:37.361000+00:00,NaN,NaN,NaN,None,None,None,c:\Projects\Github\Voyage_Analytics_MLOps_Proj...,None,8a9354e0c89303995172ede154acac958c08451e,LOCAL,auspicious-ray-883,Ganesh


In [14]:
import mlflow
import os

print("Tracking URI:", mlflow.get_tracking_uri())
print("Current Working Directory:", os.getcwd())

Tracking URI: file:///c:/Projects/Github/Voyage_Analytics_MLOps_Project/notebooks/mlruns
Current Working Directory: c:\Projects\Github\Voyage_Analytics_MLOps_Project\notebooks


In [17]:
runs = mlflow.search_runs(
    experiment_names=["Flight Price Prediction"]
)

print("Runs:", len(runs))

runs[
    [
        "run_id",
        "metrics.MAE",
        "metrics.RMSE",
        "metrics.R2"
    ]
]

Runs: 2


,run_id,metrics.MAE,metrics.RMSE,metrics.R2
0,60e284184ef247e2a86a2b2db4cddba8,2.874319e-12,4.131942e-12,1.0
1,73dd46d0be294acda437105693baf5f4,NaN,NaN,NaN


# Launch MLflow UI

Open terminal in project root:

```bash
mlflow ui
```

Open browser:

http://127.0.0.1:5000

Verify:

- Experiment: Flight Price Prediction
- Parameters
- Metrics
- Artifacts